# Audit 4, follow-up — Is the inert "response" a well-position effect? And does it survive DESeq2?

Two checks for Audit 4 (Cytokine Dictionary):
1. **Position.** (a) Does every cytokine occupy the same well on every donor's plate? (b) Are differences between inert cytokines reproducible across independent groups of donors? (c) Are inert cytokines in nearby wells (same row or column) more similar than distant ones?
2. **A standard DE method.** Repeat the DEG counts with pseudobulk **DESeq2** (`pydeseq2`), design `~ donor + condition` (paired by donor), against PBS and against the pooled other inert cytokines, for the 13 inert cytokines and 13 immune cytokines.

Uses the pseudobulk cache (`data/parse10m/cache`) and only the `obs` table of the raw file (fast).

In [1]:
# %pip install -q pydeseq2


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import numpy as np, pandas as pd, anndata as ad
from scipy.stats import spearmanr, pearsonr
from itertools import combinations
import matplotlib.pyplot as plt, seaborn as sns

PARSE = "data/parse10m/20260203_Parse_10M_PBMC_cytokines.h5ad"
CACHE_DIR = "data/parse10m/cache"
RESULTS_DIR = "results/dictionary_position"
os.makedirs(RESULTS_DIR, exist_ok=True)
CONTROL = "PBS"
INERT = ["Noggin", "Decorin", "PSPN", "GDNF", "Megalin", "PRL", "EPO", "TPO", "FGF-beta", "EGF", "HGF", "VEGF", "IGF-1"]
IMMUNE = ["IFN-beta", "IFN-gamma", "IL-2", "IL-4", "IL-7", "IL-10", "IL-15", "IL-21", "GM-CSF", "M-CSF", "IL-1-beta", "TNF-alpha", "CD40L"]
CELL_TYPES = ["B", "CD4 T", "CD8 T", "CD14 Mono", "CD16 Mono", "NK"]
WELL_COL = "bc1_well"; MIN_CELLS = 50; MIN_CPM = 10; N_SPLITS = 200; SEED = 0

sums = np.load(os.path.join(CACHE_DIR, "pb_sums.npy"))
meta = pd.read_pickle(os.path.join(CACHE_DIR, "pb_meta.pkl")).reset_index(drop=True)
key = meta[["donor", "cell_type", "cytokine"]].astype(str).agg("|".join, axis=1)
codes, uniq = pd.factorize(key)
FULL = np.zeros((len(uniq), sums.shape[1])); np.add.at(FULL, codes, sums)
NC = np.bincount(codes, weights=meta.n_cells.to_numpy(), minlength=len(uniq))
IDX = {tuple(u.split("|")): i for i, u in enumerate(uniq) if NC[i] >= MIN_CELLS}
DONORS = sorted({k[0] for k in IDX})
def log2cpm(x): return np.log2(x / max(x.sum(), 1) * 1e6 + 1)

## 1a. Plate layout: is each cytokine in the same well for every donor?

In [2]:
obs = ad.read_h5ad(PARSE, backed="r").obs
lay = (obs.groupby(["cytokine", "donor"], observed=True)[WELL_COL]
       .agg(lambda s: s.astype(str).value_counts().index[0]).unstack("donor"))
n_wells = lay.nunique(axis=1)
print(f"cytokines in the same well for all donors: {(n_wells == 1).sum()} of {len(n_wells)}")
print("cytokines with more than one well:", n_wells[n_wells > 1].to_dict())
WELL = lay.mode(axis=1)[0].astype(str)
POS = pd.DataFrame({"well": WELL, "row": WELL.str[0], "col": WELL.str[1:].astype(int)})
POS.to_csv(f"{RESULTS_DIR}/cytokine_wells.csv")
print(POS.loc[[c for c in INERT if c in POS.index]])

cytokines in the same well for all donors: 90 of 91
cytokines with more than one well: {'PBS': 6}
         well row  col
cytokine              
Noggin     F7   F    7
Decorin    F8   F    8
PSPN       F3   F    3
GDNF       F2   F    2
Megalin    E4   E    4
PRL        F4   F    4
EPO       F10   F   10
TPO        G1   G    1
FGF-beta  E11   E   11
EGF        E9   E    9
HGF       E12   E   12
VEGF       F1   F    1
IGF-1     E10   E   10


## 1b-c. Inert pseudo-responses: reproducible across donors? related to position?
Pseudo-response of an inert cytokine = its log2 CPM minus the mean of the other inert cytokines, per donor. (b) Donors are split at random into two groups of 6; the mean pseudo-responses of the two groups are correlated for the same cytokine and, as a null, for different cytokines. (c) Similarity between two inert cytokines (donor-averaged pseudo-responses) is related to their distance on the plate.

In [3]:
rng = np.random.default_rng(SEED)
rows_b, rows_c = [], []
for ct in CELL_TYPES:
    prof = {(d, c): log2cpm(FULL[IDX[(d, ct, c)]]) for d in DONORS for c in INERT + [CONTROL] if (d, ct, c) in IDX}
    pbs = np.mean([prof[(d, CONTROL)] for d in DONORS if (d, CONTROL) in prof], 0)
    expressed = (2 ** pbs - 1) >= MIN_CPM
    inert = [c for c in INERT if sum((d, c) in prof for d in DONORS) >= 10]
    PR = {}
    for c in inert:
        for d in DONORS:
            others = [prof[(d, k)] for k in inert if k != c and (d, k) in prof]
            if (d, c) in prof and len(others) >= 3:
                PR[(d, c)] = (prof[(d, c)] - np.mean(others, 0))[expressed]
    # (b) reproducibility across donor halves
    same, diff = [], []
    for _ in range(N_SPLITS):
        g1 = set(rng.choice(DONORS, len(DONORS) // 2, replace=False))
        m = {}
        for c in inert:
            a = [PR[(d, c)] for d in DONORS if (d, c) in PR and d in g1]
            b = [PR[(d, c)] for d in DONORS if (d, c) in PR and d not in g1]
            if len(a) >= 3 and len(b) >= 3: m[c] = (np.mean(a, 0), np.mean(b, 0))
        cs = list(m)
        same += [pearsonr(m[c][0], m[c][1])[0] for c in cs]
        diff += [pearsonr(m[c][0], m[k][1])[0] for c in cs for k in cs if k != c]
    rows_b.append(dict(cell_type=ct, same_cytokine=np.median(same), different_cytokines=np.median(diff)))
    # (c) similarity vs plate distance
    avg = {c: np.mean([PR[(d, c)] for d in DONORS if (d, c) in PR], 0) for c in inert}
    for a, b in combinations([c for c in inert if c in POS.index], 2):
        pa, pb = POS.loc[a], POS.loc[b]
        rows_c.append(dict(cell_type=ct, a=a, b=b, r=pearsonr(avg[a], avg[b])[0],
                           same_row=pa.row == pb.row, same_col=pa.col == pb.col,
                           distance=np.hypot(ord(pa.row) - ord(pb.row), pa.col - pb.col)))
REPRO = pd.DataFrame(rows_b).set_index("cell_type").round(3)
REPRO.to_csv(f"{RESULTS_DIR}/inert_reproducibility.csv")
print("(b) Inert pseudo-responses, correlation between two independent donor groups"); display(REPRO)

NEAR = pd.DataFrame(rows_c); NEAR.to_csv(f"{RESULTS_DIR}/inert_similarity_vs_position.csv", index=False)
def mantel(g, n=2000):
    obs_rho = spearmanr(g.distance, g.r)[0]
    names = sorted(set(g.a) | set(g.b)); null = []
    for _ in range(n):
        perm = dict(zip(names, rng.permutation(names)))
        dist = [np.hypot(ord(POS.loc[perm[a]].row) - ord(POS.loc[perm[b]].row), POS.loc[perm[a]].col - POS.loc[perm[b]].col)
                for a, b in zip(g.a, g.b)]
        null.append(spearmanr(dist, g.r)[0])
    return obs_rho, (1 + np.sum(np.array(null) <= obs_rho)) / (1 + n)
rows = []
for ct, g in NEAR.groupby("cell_type"):
    rho, p = mantel(g, 500)
    rows.append(dict(cell_type=ct, rho_similarity_vs_distance=rho, p_mantel=p,
                     r_same_row=g[g.same_row].r.mean(), r_same_col=g[g.same_col].r.mean(),
                     r_other=g[~g.same_row & ~g.same_col].r.mean()))
POSTEST = pd.DataFrame(rows).set_index("cell_type").round(3)
POSTEST.to_csv(f"{RESULTS_DIR}/inert_position_test.csv")
print("(c) Similarity of inert pseudo-responses vs plate distance (negative rho = nearby wells more similar)"); display(POSTEST)

(b) Inert pseudo-responses, correlation between two independent donor groups


,same_cytokine,different_cytokines
cell_type,,
B,0.540,-0.043
CD4 T,0.786,-0.068
CD8 T,0.648,-0.055
CD14 Mono,0.694,-0.062
CD16 Mono,0.370,-0.029
NK,0.397,-0.032


(c) Similarity of inert pseudo-responses vs plate distance (negative rho = nearby wells more similar)


,rho_similarity_vs_distance,p_mantel,r_same_row,r_same_col,r_other
cell_type,,,,,
B,-0.015,0.385,-0.115,-0.067,-0.059
CD14 Mono,-0.018,0.343,-0.122,0.002,-0.058
CD16 Mono,-0.011,0.377,-0.110,-0.047,-0.065
CD4 T,-0.028,0.329,-0.124,-0.041,-0.052
CD8 T,-0.004,0.371,-0.122,-0.046,-0.055
NK,0.019,0.485,-0.113,-0.060,-0.062


## 2. DEG counts with pseudobulk DESeq2
For each cell type and tested cytokine: counts of the cytokine and of the reference for every donor, design `~ donor + condition`, Wald test, padj < 0.05 and |log2FC| > 0.25 (the paper's thresholds). References: PBS, and the pooled counts of the other inert cytokines (a single reference sample per donor). Genes: expressed in PBS (CPM >= 10).

**Runtime:** 26 cytokines x 2 references x 6 cell types = 312 DESeq2 fits; roughly 1-2 h depending on cores.

In [4]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference
INF = DefaultInference(n_cpus=16)          # set to the number of CPU cores you want to use
import logging, warnings; logging.getLogger("pydeseq2").setLevel(logging.ERROR); warnings.filterwarnings("ignore")

def deseq_ndeg(counts_c, counts_r, donors):
    X = np.vstack([counts_c, counts_r]).round().astype(int)
    md_ = pd.DataFrame({"donor": donors + donors, "condition": ["treat"] * len(donors) + ["ref"] * len(donors)},
                       index=[f"s{i}" for i in range(2 * len(donors))])
    dds = DeseqDataSet(counts=pd.DataFrame(X, index=md_.index), metadata=md_, design="~donor + condition", quiet=True, inference=INF)
    dds.deseq2()
    st = DeseqStats(dds, contrast=["condition", "treat", "ref"], quiet=True, inference=INF); st.summary()
    r = st.results_df
    return int(((r.padj < 0.05) & (r.log2FoldChange.abs() > 0.25)).sum())

rows = []
for ct in CELL_TYPES:
    pbs_tot = np.sum([FULL[IDX[(d, ct, CONTROL)]] for d in DONORS if (d, ct, CONTROL) in IDX], 0)
    expressed = pbs_tot / pbs_tot.sum() * 1e6 >= MIN_CPM
    for c in INERT + IMMUNE:
        for ref in ["PBS", "inert"]:
            cc, rr, dd = [], [], []
            for d in DONORS:
                if (d, ct, c) not in IDX: continue
                if ref == "PBS":
                    if (d, ct, CONTROL) not in IDX: continue
                    r = FULL[IDX[(d, ct, CONTROL)]]
                else:
                    others = [FULL[IDX[(d, ct, k)]] for k in INERT if k != c and (d, ct, k) in IDX]
                    if len(others) < 3: continue
                    r = np.sum(others, 0)
                cc.append(FULL[IDX[(d, ct, c)]][expressed]); rr.append(r[expressed]); dd.append(d)
            if len(dd) < 6: continue
            rows.append(dict(cell_type=ct, cytokine=c, inert=c in INERT, reference=ref, n_donors=len(dd),
                             n_deg=deseq_ndeg(np.array(cc), np.array(rr), dd)))
    print(ct, "done")
DESEQ = pd.DataFrame(rows); DESEQ.to_csv(f"{RESULTS_DIR}/deseq2_deg_counts.csv", index=False)
DESEQ_SUMMARY = DESEQ.groupby(["reference", "inert"]).n_deg.agg(["count", "median", "mean", lambda x: (x > 0).mean()]).round(2)
DESEQ_SUMMARY.columns = ["n", "median_deg", "mean_deg", "share_with_deg"]
DESEQ_SUMMARY.to_csv(f"{RESULTS_DIR}/deseq2_summary.csv")
display(DESEQ_SUMMARY)
display(DESEQ.pivot_table(index="cell_type", columns=["reference", "inert"], values="n_deg", aggfunc="median"))

B done
CD4 T done
CD8 T done
CD14 Mono done
CD16 Mono done
NK done


n  median_deg  mean_deg  share_with_deg
reference inert                                          
PBS       False  78      2241.0   2286.09             1.0
          True   78       414.5    484.94             1.0
inert     False  78      2383.0   2355.06             1.0
          True   78       375.5    414.55             1.0

reference     PBS          inert       
inert       False  True    False  True 
cell_type                              
B          2130.0  516.0  2267.0  474.0
CD14 Mono  3355.0  482.0  3515.0  392.0
CD16 Mono  2641.0  338.0  2885.0  281.0
CD4 T      2064.0  517.0  2087.0  414.0
CD8 T      1790.0  545.0  1786.0  462.0
NK         1602.0  317.0  1401.0  326.0

In [5]:
STRICT = ["TPO", "Megalin", "PSPN", "EPO", "EGF"]
D = pd.read_csv(f"{RESULTS_DIR}/deseq2_deg_counts.csv")
D = D[(D.reference == "PBS") & (~D.inert | D.cytokine.isin(STRICT))]
D["group"] = np.where(D.cytokine.isin(STRICT), "strict inert", "immune")
display(D.groupby("group").n_deg.agg(["count", "median", "mean", lambda x: (x > 0).mean()]).round(1))
display(D.pivot_table(index="cell_type", columns="group", values="n_deg", aggfunc="median"))

,count,median,mean,<lambda_0>
group,,,,
immune,78,2241.0,2286.1,1.0
strict inert,30,389.0,407.5,1.0


group,immune,strict inert
cell_type,,
B,2130.0,399.0
CD14 Mono,3355.0,471.0
CD16 Mono,2641.0,353.0
CD4 T,2064.0,387.0
CD8 T,1790.0,401.0
NK,1602.0,269.0
